# 04 · What does the evidence permit next?


The final question is whether the full comparison justifies continuing. Read
completion, validity and effect size separately. A failed job, a failed
causality audit and a valid negative prediction result require different next
actions even when a saved report labels each one STOP.

In `execute` mode this notebook scores and builds the report. `inspect` mode
only reads it. Both use the experiment's existing decision rules.

**Run this notebook independently in a fresh kernel.** The default
`teach` mode uses small generated examples. Set `FI_TUTORIAL_MODE=inspect`
and `FI_RUN_ROOT` before starting the kernel to read saved artifacts.
Set `FI_TUTORIAL_MODE=execute` with an explicit `FI_RUN_ROOT` to run the
production stages below. Execute notebooks **00 → 04** in order for the
full Experiment 0; each uses a fresh kernel and the same run directory.
Use the [notebook HAIC launchers](../../../slurm/future-innovation/NOTEBOOKS.md)
for scheduled execution. Inspection remains read-only. An absent local
file says nothing about the current state of a remote HAIC job.

[Study overview](../../../docs/studies/future-innovation/README.md) ·
[Direct gate specification](../../../docs/studies/future-innovation/direct-gate-protocol.md)

In [ ]:
from pathlib import Path
import os
import sys
from time import perf_counter

started = perf_counter()
override = os.environ.get("GAVD6_ROOT")
if override:
    candidates = [Path(override).expanduser().resolve()]
else:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base, base / "gavd6", base / "experiments/sjepa/gavd6"))
PROJECT_ROOT = next((p for p in candidates if (p / "src/gavd6_sjepa").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set GAVD6_ROOT to the checkout containing src/gavd6_sjepa.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from matplotlib_inline.backend_inline import set_matplotlib_formats
get_ipython().run_line_magic("matplotlib", "inline")
set_matplotlib_formats("svg", "png")
plt.rcParams.update({"figure.figsize": (8, 3), "axes.spines.top": False,
                    "axes.spines.right": False, "font.size": 11})

from gavd6_sjepa.research_directions.future_innovation.fi_tutorial_inspection import (
    artifact_inventory, inspect_report, read_optional_table, inspection_audit_path,
)

# Use "execute" for real stages or "inspect" for saved artifacts.
# Relative paths resolve from GAVD6_ROOT. Execution requires an explicit run root.
MODE = os.environ.get("FI_TUTORIAL_MODE", "teach")
if MODE not in {"teach", "inspect", "execute"}:
    raise ValueError("FI_TUTORIAL_MODE must be teach, inspect, or execute.")
if MODE == "execute" and not os.environ.get("FI_RUN_ROOT"):
    raise ValueError("Set FI_RUN_ROOT explicitly before executing real experiment stages.")
RUN_ROOT = Path(os.environ.get("FI_RUN_ROOT", "outputs/future-innovation/direct-v2")).expanduser()
if not RUN_ROOT.is_absolute():
    RUN_ROOT = PROJECT_ROOT / RUN_ROOT
RUN_ROOT = RUN_ROOT.resolve()
print("Teaching examples only; no empirical gait findings." if MODE == "teach"
      else f"{MODE.upper()} mode: {RUN_ROOT}")
if MODE == "execute":
    from gavd6_sjepa.research_directions.future_innovation.fi_notebook_workflow import (
        initialize_from_environment, run_stage, build_notebook_report, finish_notebook_report,
        attempt_stage, require_stage_success,
    )
    if (RUN_ROOT / "config/run-contract.json").is_file():
        import json
        saved_run = json.loads((RUN_ROOT / "config/run-contract.json").read_text())
        print("Frozen protocol:", saved_run.get("protocol", "legacy-v1"),
              "— gate clips:", saved_run.get("cohort_size"))
        if saved_run.get("protocol", "legacy-v1") == "legacy-v1":
            print("This run retains legacy selectivity gates. Use a new run root for direct-v2.")

## Execute this stage

Score the out-of-fold predictions and build the sealed production report. If scoring fails, still attempt a diagnostic STOP report. A verified audit rejection builds an unsealed diagnostic STOP and finishes with TRAINING BLOCKED, measurement_complete=False. Unexpected scoring failures and missing or corrupt evidence still fail. Complete STOP and INCONCLUSIVE results are also successful executions.

This cell runs only in `execute` mode. Each command uses this kernel's Python and the existing production CLI; stage logs are retained alongside the executed notebook.

In [ ]:
if MODE == "execute":
    scoring_succeeded = build_notebook_report(RUN_ROOT)

## 1. Understand the decision rule with invented evidence

The production gate first checks valid finite measurements, then point
thresholds, then source-bootstrap and seed stability. These invented
cases show its behavior. An illustrative ADVANCE has no authority over
a real run. Adapter training is disallowed even when this raw-skeleton
feasibility gate advances.

In [ ]:
if MODE == "teach":
    from gavd6_sjepa.research_directions.future_innovation.fi_gate_decision import decide_gate
    metrics = {
        "delta_r2_real": 0.06, "delta_r2_time_shuffle": 0.02,
        "delta_r2_clip_mismatch": 0.005,
        "delta_r2_no_skeleton": 0.005, "seed_skeleton_increments": [0.055] * 3,
        "skeleton_increment_positive_fraction": 0.95, "bootstrap_positive_fraction": 0.95,
        "seed_real_gains": [0.06, 0.06, 0.06],
        **{key: True for key in ("data_contract_valid", "evaluation_contract_valid",
            "controls_complete", "input_audit_complete", "target_variance_valid",
            "teacher_stable", "causal_leakage_absent")},
    }
    scenarios = {
        "Illustrative stable gain": metrics,
        "Illustrative weak real gain": {**metrics, "delta_r2_real": 0.01, "seed_real_gains": [0.01] * 3, "seed_skeleton_increments": [0.005] * 3},
        "Extra capacity explains gain": {**metrics, "delta_r2_no_skeleton": 0.06, "seed_skeleton_increments": [0.0] * 3},
        "Illustrative uncertainty": {**metrics, "bootstrap_positive_fraction": 0.7},
        "Illustrative invalid input": {**metrics, "causal_leakage_absent": False},
    }
    display(pd.DataFrame([{"invented case": name, "decision": decide_gate(values, protocol="direct-v2")["decision"],
        "adapter allowed": decide_gate(values, protocol="direct-v2")["allow_adapter_training"]}
        for name, values in scenarios.items()]))

## 2. Establish the status of the saved evidence

A local copy may contain only overlays or an incomplete STOP report.
A complete report is sealed with hashes for its decision and narrative.
Inspection uses the production seal verifier and never rewrites it.
A seal establishes file integrity; it does not independently rerun the
original data, causal audits, predictions or statistical analysis.

| Inspection state | Interpretation |
| --- | --- |
| Unavailable | No decision in this copy; remote state is unknown |
| Incomplete | Execution or measurement still needs repair |
| Invalid / unverified | Resolve missing or inconsistent report evidence |
| Synthetic | Software demonstration; no scientific advancement |
| Complete / STOP | Read effect and control checks; the validity audits passed before fitting |
| Complete / INCONCLUSIVE | Point checks passed; stability was insufficient |
| Complete / ADVANCE | Design the full-GAVD JEPA comparison; adapter distillation remains separate |

In [ ]:
if MODE != "teach":
    evidence = inspect_report(RUN_ROOT)
    print(evidence["state"], "—", evidence["explanation"])
    decision = evidence["decision"]
    if decision is not None:
        print("Saved reason:", decision.get("reason", "No reason recorded"))
        display(pd.DataFrame(list(decision.get("checks", {}).items()), columns=["saved check", "passed"]))
    if evidence["report_text"] is not None:
        display(Markdown(evidence["report_text"]))

## 3. Compare controls before attributing a gain to motion

Pool out-of-fold predictions within each seed, then average seed scores.
Do not average fold R² values or treat repeated seeds as new people.
Source bootstraps keep each source's windows together. Intervals are
conditional on the saved fits; model selection and training are not
repeated within each draw. The sealed report
includes the per-seed scores, intervals and paired real-minus-control
contrasts; use that report for numerical interpretation.

The primary skeleton increment is the real head's R² minus the matched
no-skeleton head's R². That control retains time-varying validity flags
and receives the same RGB and nuisance inputs, so the contrast measures
additional coordinate/confidence history. A positive mean must also
appear in every seed and at least 90% of paired source draws to advance.
Report its 95% interval separately; crossing zero leaves uncertainty.
The real head must still improve on ridge by at least 0.05, exceed twice
the nonnegative shuffled gain, and keep wrong-clip gain at most 0.01.

The target remains contextual and its horizon is located inside
full-clip teacher features. No result here establishes a clinical
endpoint, isolated dynamics, or the benefit of a trained S-JEPA student.

## 4. Record one next action

| Observation | Next action |
| --- | --- |
| Missing or incomplete artifacts | Recover the existing run through its normal pipeline |
| Failed input or teacher validity | Repair the measurement before interpreting prediction scores |
| Complete, valid gain below the rule | Preserve the negative result; reconsider the representation or endpoint |
| Gain reproduced by controls | Investigate the remaining shortcut or capacity explanation |
| Unstable gain | Report uncertainty and define any further measurement before running it |
| Complete stable skeleton increment | Compare trained JEPA features, matched initial encoders and raw skeleton history |

Update the study overview with the identified run, exact report path,
supported conclusion, unresolved explanation, and next action. Keep
synthetic examples and older prompts visibly separate from that record.
Preserve the executed notebook as an artifact when adding result-specific
commentary; edit the builder for changes to the reusable explanation.

In [ ]:
if MODE == "execute":
    completed_decision = finish_notebook_report(RUN_ROOT, scoring_succeeded=scoring_succeeded)

## What this step establishes

Record the identified run, its measurement status, the supported conclusion and one next action. Execution scores and seals complete evidence; inspection reads the existing seal. A complete negative result is useful evidence, while an incomplete STOP calls for execution or measurement repair.

Return to the [study overview](../../../docs/studies/future-innovation/README.md) to record the next decision.

In [ ]:
print(f"Notebook elapsed time: {perf_counter() - started:.2f} seconds ({MODE} mode).")